In [1]:
!pip install -q ultralytics gradio

from google.colab import drive
drive.mount('/content/drive')

!cp "/content/drive/MyDrive/proyecto_epp/archive.zip" /content/archive.zip
!unzip -q -o /content/archive.zip -d /content/epp

yaml_text = """
path: /content/epp/css-data
train: train/images
val: valid/images
test: test/images
names:
  0: Hardhat
  1: Mask
  2: NO-Hardhat
  3: NO-Mask
  4: NO-Safety Vest
  5: Person
  6: Safety Cone
  7: Safety Vest
  8: machinery
  9: vehicle
"""
open("/content/epp/css-data/data.yaml", "w").write(yaml_text)

import torch
assert torch.cuda.is_available(), "No hay GPU: ve a Entorno de ejecución > Cambiar tipo > GPU T4"
print("GPU:", torch.cuda.get_device_name(0))

from ultralytics import YOLO
modelo = YOLO("yolov8n.pt")
modelo.train(
    data="/content/epp/css-data/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    project="/content/drive/MyDrive/proyecto_epp/resultados",
    name="v1",
)
print("¡Entrenamiento finalizado y guardado en Drive!")

Mounted at /content/drive
GPU: Tesla T4
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/epp/css-data/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flip

In [ ]:
import os, re, sys, time, shutil, subprocess, urllib.request, base64
from google.colab import drive

drive.mount("/content/drive")

VERSION = "v1"
IMGSZ = 416
CONFIANZA = 40

WEB = "/content/web"
os.makedirs(WEB, exist_ok=True)

logo_path = "/content/drive/MyDrive/proyecto_epp/ULIMA.png"
if not os.path.exists(logo_path):
    logo_path = "ULIMA.png"

logo_base64 = ""
if os.path.exists(logo_path):
    with open(logo_path, "rb") as f:
        logo_base64 = base64.b64encode(f.read()).decode("utf-8")

HTML = r'''<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Detector de EPP</title>
<script src="https://cdn.jsdelivr.net/npm/onnxruntime-web@1.20.1/dist/ort.min.js"></script>
<style>
  body{font-family:system-ui,sans-serif;background:#111;color:#eee;max-width:900px;margin:auto;padding:12px}
  .header-container{display:flex;justify-content:space-between;align-items:center;margin-bottom:8px;padding-top:4px}
  h1{font-size:1.3rem;margin:0}
  .logo-img{height:56px;width:auto;object-fit:contain;margin-top:6px;margin-right:12px}
  canvas{width:100%;border-radius:8px;background:#000}
  button,.btn{display:inline-block;padding:10px 14px;font-size:1rem;border:0;border-radius:8px;background:#e8590c;color:#fff;margin:4px 4px 4px 0;cursor:pointer}
  #banner{margin:10px 0 6px;padding:10px 12px;border-radius:8px;font-weight:bold;background:#222}
  #banner.ok{background:#12391c;color:#51cf66}
  #banner.alerta{background:#4a1515;color:#ff6b6b}
  #banner.parcial{background:#4a3a10;color:#ffd43b}
  .fila{display:flex;justify-content:space-between;align-items:center;padding:8px 12px;margin:4px 0;border-radius:8px;background:#1c1c1c}
  .fila b.bien{color:#51cf66} .fila b.mal{color:#ff6b6b} .fila b.neutro{color:#ffd43b}
  #info{font-size:.85rem;color:#999;margin:4px 0 8px}
</style>
</head>
<body>
<div class="header-container">
  <h1>Detector de Equipos de Protección Personal</h1>
  <img src="data:image/png;base64,__LOGO_BASE64__" alt="Logo" class="logo-img">
</div>
<p><b>Cómo leer el resultado:</b> 🟩 Verde = equipo puesto · 🟥 Rojo = falta el equipo.
El porcentaje indica qué tan seguro está el sistema de cada detección. Solo se muestran las detecciones con al menos __UMBRAL__% de seguridad.</p>
<div>
  <button id="btnIniciar">🎥 Cámara en vivo</button>
  <label class="btn">📷 Subir imagen<input id="archivo" type="file" accept="image/*" hidden></label>
  <button id="btnCambiar">🔄 Cambiar cámara</button>
  <button id="btnEspejo">↔ Voltear imagen</button>
</div>

<div id="banner">Cargando modelo...</div>
<div class="fila"><span>🪖 Casco</span><b id="estCasco" class="neutro">—</b></div>
<div class="fila"><span>🦺 Chaleco</span><b id="estChaleco" class="neutro">—</b></div>
<div class="fila"><span>😷 Mascarilla</span><b id="estMascarilla" class="neutro">—</b></div>
<div id="info"></div>

<canvas id="lienzo"></canvas>
<video id="video" playsinline muted style="position:absolute;width:1px;height:1px;opacity:0"></video>

<script>
const MODELO = "best.onnx";
const S = __IMGSZ__;
const NC = 10;
const UMBRAL = __UMBRAL__ / 100;
const CASCO = 0, MASCARILLA = 1, SIN_CASCO = 2, SIN_MASCARILLA = 3, SIN_CHALECO = 4, CHALECO = 7;
const NOMBRES = {0: "Casco", 1: "Mascarilla", 2: "SIN casco", 3: "SIN mascarilla", 4: "SIN chaleco", 7: "Chaleco"};
const COLOR = {0: "#00aa00", 1: "#00aa00", 7: "#00aa00", 2: "#dc0000", 3: "#dc0000", 4: "#dc0000"};
const CLASES = [CASCO, MASCARILLA, SIN_CASCO, SIN_MASCARILLA, SIN_CHALECO, CHALECO];

const video = document.getElementById("video");
const lienzo = document.getElementById("lienzo");
const ctx = lienzo.getContext("2d");
const banner = document.getElementById("banner");
const estCasco = document.getElementById("estCasco");
const estChaleco = document.getElementById("estChaleco");
const estMascarilla = document.getElementById("estMascarilla");
const info = document.getElementById("info");
const trabajo = document.createElement("canvas");
trabajo.width = S; trabajo.height = S;
const tctx = trabajo.getContext("2d", { willReadFrequently: true });

let sesion = null, flujo = null, ocupado = false, activo = false;
let camara = "environment", detecciones = [], fps = 0;
let modo = "vivo", imagen = null, espejo = false;

function mensaje(texto, clase) {
  banner.textContent = texto;
  banner.className = clase || "";
}

function origen() {
  if (modo === "foto" && imagen) return { el: imagen, w: imagen.naturalWidth, h: imagen.naturalHeight };
  return { el: video, w: video.videoWidth, h: video.videoHeight };
}

async function cargarModelo() {
  if (typeof ort === "undefined") {
    mensaje("⚠️ No se pudo cargar el motor de IA.", "alerta");
    return;
  }
  try {
    ort.env.wasm.wasmPaths = "https://cdn.jsdelivr.net/npm/onnxruntime-web@1.20.1/dist/";
    sesion = await ort.InferenceSession.create(MODELO, { executionProviders: ["wasm"] });
    mensaje("Modelo listo. Presiona «Cámara en vivo».");
  } catch (e) {
    mensaje("Error al cargar el modelo: " + e, "alerta");
  }
}

async function iniciarCamara() {
  try {
    if (flujo) flujo.getTracks().forEach(t => t.stop());
    flujo = await navigator.mediaDevices.getUserMedia({
      video: { facingMode: camara, width: { ideal: 426 }, height: { ideal: 240 } }, audio: false });
    video.srcObject = flujo;
    await video.play();
    modo = "vivo";
    detecciones = [];
    lienzo.width = video.videoWidth;
    lienzo.height = video.videoHeight;
    if (!activo) { activo = true; requestAnimationFrame(pintar); }
  } catch (e) {
    mensaje("No se pudo abrir la cámara: " + e, "alerta");
  }
}

function pintar() {
  if (modo === "vivo" && video.videoWidth) {
    redibujar();
    if (!ocupado && sesion) inferir();
  }
  requestAnimationFrame(pintar);
}

function redibujar() {
  const f = origen();
  if (!f.w) return;
  const k = lienzo.width / f.w;
  ctx.save();
  if (espejo) { ctx.translate(lienzo.width, 0); ctx.scale(-1, 1); }
  ctx.drawImage(f.el, 0, 0, lienzo.width, lienzo.height);
  ctx.restore();
  dibujarCajas(k);
}

async function inferir() {
  ocupado = true;
  const t0 = performance.now();
  try {
    detecciones = await detectar();
    fps = 1000 / (performance.now() - t0);
    actualizarEstado();
  } catch (e) {
    mensaje("Error: " + e, "alerta");
  }
  ocupado = false;
}

async function analizarFoto() {
  if (!sesion) { mensaje("El modelo aún no está listo."); return; }
  while (ocupado) await new Promise(r => setTimeout(r, 50));
  await inferir();
  redibujar();
}

async function detectar() {
  const f = origen();
  const escala = Math.min(S / f.w, S / f.h);
  const nw = Math.round(f.w * escala), nh = Math.round(f.h * escala);
  const px = Math.floor((S - nw) / 2), py = Math.floor((S - nh) / 2);

  tctx.fillStyle = "rgb(114,114,114)";
  tctx.fillRect(0, 0, S, S);
  tctx.drawImage(f.el, px, py, nw, nh);
  const pix = tctx.getImageData(0, 0, S, S).data;

  const n = S * S, entrada = new Float32Array(3 * n);
  for (let i = 0; i < n; i++) {
    entrada[i] = pix[i * 4] / 255;
    entrada[n + i] = pix[i * 4 + 1] / 255;
    entrada[2 * n + i] = pix[i * 4 + 2] / 255;
  }
  const res = await sesion.run({ [sesion.inputNames[0]]: new ort.Tensor("float32", entrada, [1, 3, S, S]) });
  const out = res[sesion.outputNames[0]];
  const d = out.data, N = out.dims[2], umbral = UMBRAL, cajas = [];
  for (let i = 0; i < N; i++) {
    let mejor = -1, mc = -1;
    for (const c of CLASES) {
      const s = d[(4 + c) * N + i];
      if (s > mejor) { mejor = s; mc = c; }
    }
    if (mejor < umbral) continue;
    const cx = d[i], cy = d[N + i], w = d[2 * N + i], h = d[3 * N + i];
    cajas.push({ c: mc, s: mejor,
      x1: (cx - w / 2 - px) / escala, y1: (cy - h / 2 - py) / escala,
      x2: (cx + w / 2 - px) / escala, y2: (cy + h / 2 - py) / escala });
  }
  return resolverConflictos(nms(cajas, 0.45));
}

function iou(a, b) {
  const x1 = Math.max(a.x1, b.x1), y1 = Math.max(a.y1, b.y1);
  const x2 = Math.min(a.x2, b.x2), y2 = Math.min(a.y2, b.y2);
  const inter = Math.max(0, x2 - x1) * Math.max(0, y2 - y1);
  const union = (a.x2 - a.x1) * (a.y2 - a.y1) + (b.x2 - b.x1) * (b.y2 - b.y1) - inter;
  return inter / union;
}

function nms(cajas, umbral) {
  cajas.sort((a, b) => b.s - a.s);
  const res = [];
  for (const c of cajas) {
    if (res.every(r => r.c !== c.c || iou(r, c) < umbral)) res.push(c);
  }
  return res;
}

const OPUESTO = {0: 2, 2: 0, 7: 4, 4: 7, 1: 3, 3: 1};
function resolverConflictos(cajas) {
  return cajas.filter(a => !cajas.some(b =>
    b.c === OPUESTO[a.c] && iou(a, b) > 0.4 && b.s > a.s));
}

function dibujarCajas(k) {
  const W = lienzo.width, H = lienzo.height;
  const tam = Math.max(12, Math.round(W / 42));
  ctx.lineWidth = Math.max(2, W / 300);
  ctx.font = `bold ${tam}px sans-serif`;
  ctx.textBaseline = "top";

  const cajas = detecciones.map(d => {
    let x1 = d.x1 * k, x2 = d.x2 * k;
    if (espejo) { const t = x1; x1 = W - x2; x2 = W - t; }
    return { d, x1, x2, y1: d.y1 * k, y2: d.y2 * k };
  });

  for (const b of cajas) {
    ctx.strokeStyle = COLOR[b.d.c];
    ctx.strokeRect(b.x1, b.y1, b.x2 - b.x1, b.y2 - b.y1);
  }

  const ocupados = [];
  const choca = r => ocupados.some(o =>
    r.x < o.x + o.w && o.x < r.x + r.w && r.y < o.y + o.h && o.y < r.y + r.h);
  cajas.sort((a, b) => b.d.s - a.d.s);
  for (const b of cajas) {
    const txt = `${NOMBRES[b.d.c]} ${Math.round(b.d.s * 100)}%`;
    const w = ctx.measureText(txt).width + 8, h = tam + 6;
    const x = Math.min(Math.max(0, b.x1), Math.max(0, W - w));
    const intentos = [b.y1 - h, b.y2, b.y1, b.y1 + h, b.y2 + h, b.y1 - 2 * h, b.y1 + 2 * h, b.y2 - h];
    let r = null;
    for (const y of intentos) {
      const c = { x, y: Math.min(Math.max(0, y), H - h), w, h };
      if (!choca(c)) { r = c; break; }
    }
    if (!r) r = { x, y: Math.min(Math.max(0, b.y1 - h), H - h), w, h };
    ocupados.push(r);
    ctx.fillStyle = COLOR[b.d.c];
    ctx.fillRect(r.x, r.y, r.w, r.h);
    ctx.fillStyle = "#fff";
    ctx.fillText(txt, r.x + 4, r.y + 3);
  }
}

function contar(c) { return detecciones.filter(d => d.c === c).length; }

function estadoEquipo(el, nombre, ok, falta, fem) {
  const N = nombre.toUpperCase();
  const fin = fem ? "A" : "O";
  let texto, clase;
  if (falta > 0) {
    texto = `❌ SIN ${N}` + (falta > 1 ? ` (${falta})` : "");
    if (ok > 0) texto += ` · con ${nombre.toLowerCase()}: ${ok}`;
    clase = "mal";
  } else if (ok > 0) {
    texto = `✅ ${N} DETECTAD${fin}` + (ok > 1 ? ` (${ok})` : "");
    clase = "bien";
  } else {
    texto = `➖ ${N} NO DETECTAD${fin}`;
    clase = "neutro";
  }
  el.textContent = texto;
  el.className = clase;
}

function actualizarEstado() {
  const cOk = contar(CASCO), cNo = contar(SIN_CASCO);
  const vOk = contar(CHALECO), vNo = contar(SIN_CHALECO);
  const mOk = contar(MASCARILLA), mNo = contar(SIN_MASCARILLA);
  estadoEquipo(estCasco, "Casco", cOk, cNo, false);
  estadoEquipo(estChaleco, "Chaleco", vOk, vNo, false);
  estadoEquipo(estMascarilla, "Mascarilla", mOk, mNo, true);

  const faltan = [];
  if (cOk === 0) faltan.push("el casco");
  if (vOk === 0) faltan.push("el chaleco");
  if (mOk === 0) faltan.push("la mascarilla");

  if (cNo > 0 || vNo > 0 || mNo > 0) {
    mensaje("⚠️ ALERTA: falta equipo de protección", "alerta");
  } else if (faltan.length === 0) {
    mensaje("✅ EPP COMPLETO", "ok");
  } else {
    mensaje("ℹ️ Equipo incompleto", "parcial");
  }
  info.textContent = modo === "vivo" ? `${fps.toFixed(1)} análisis/s` : "";
}

document.getElementById("btnIniciar").onclick = iniciarCamara;
document.getElementById("btnCambiar").onclick = () => {
  camara = camara === "user" ? "environment" : "user";
  iniciarCamara();
};
document.getElementById("btnEspejo").onclick = () => { espejo = !espejo; };
document.getElementById("archivo").onchange = (e) => {
  const archivo = e.target.files[0];
  if (!archivo) return;
  const img = new Image();
  img.onload = async () => {
    imagen = img;
    modo = "foto";
    detecciones = [];
    const k = Math.min(1, 1000 / Math.max(img.naturalWidth, img.naturalHeight));
    lienzo.width = Math.round(img.naturalWidth * k);
    lienzo.height = Math.round(img.naturalHeight * k);
    redibujar();
    await analizarFoto();
  };
  img.src = URL.createObjectURL(archivo);
  e.target.value = "";
};
cargarModelo();
</script>
</body>
</html>
'''

HTML = HTML.replace("__IMGSZ__", str(IMGSZ))\
           .replace("__UMBRAL__", str(CONFIANZA))\
           .replace("__LOGO_BASE64__", logo_base64)

with open(f"{WEB}/index.html", "w", encoding="utf-8") as f:
    f.write(HTML)

PESOS = f"/content/drive/MyDrive/proyecto_epp/resultados/{VERSION}/weights"
onnx_drive = f"{PESOS}/best_{IMGSZ}.onnx"
if not os.path.exists(onnx_drive):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "onnx", "onnxruntime"], check=True)
    from ultralytics import YOLO
    YOLO(f"{PESOS}/best.pt").export(format="onnx", imgsz=IMGSZ, opset=12)
    shutil.move(f"{PESOS}/best.onnx", onnx_drive)
shutil.copy(onnx_drive, f"{WEB}/best.onnx")

for viejo in ["http.server 8000", "cloudflared", "ngrok"]:
    subprocess.run(["pkill", "-f", viejo])
time.sleep(1)
subprocess.Popen([sys.executable, "-m", "http.server", "8000", "--directory", WEB],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)

CF = "/content/cloudflared"
if not os.path.exists(CF):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CF)
    os.chmod(CF, 0o755)

log = open("/content/cf.log", "w")
tunel = subprocess.Popen([CF, "tunnel", "--no-autoupdate", "--protocol", "http2", "--url", "http://localhost:8000"], stdout=log, stderr=subprocess.STDOUT)

enlace = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("/content/cf.log").read())
    if m:
        enlace = m.group(0)
        break
    if tunel.poll() is not None:
        break

print("\n" + "=" * 62)
print("Enlace disponible:")
print(enlace)
print("=" * 62)

try:
    while True:
        time.sleep(30)
        if tunel.poll() is not None:
            break
except KeyboardInterrupt:
    print("Celda detenida.")

Mounted at /content/drive
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
Model summary (fused): 73 layers, 3,134,894 parameters, 0 gradients, 3.7 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/proyecto_epp/resultados/v1/weights/best.pt' with input shape (1, 3, 416, 416) BCHW and output shape(s) (1, 14, 3549) (30.3 MB)
requirements: Ultralytics requirement ['onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.16 environment at: /usr
Resolved 10 packages in 341ms
Prepared 2 packages in 36ms
Inst